In [2]:
import os
import glob
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # needed for full CUDA determinism
from pathlib import Path
import random
import numpy as np
import torch
import torch.backends.cudnn as cudnn
import os
import csv
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader
import torchvision
import torchvision.transforms as transforms
from util import process_yaml_file, process_experiment_setup, create_train_subset, evaluate2, prepare_group_subset, \
                 process_experiment_setup_deit, create_or_load_group_A, create_class_balanced_mix_train_test, MixedSplitDataset, \
                 load_best_checkpoint

# =====================================================
# 1. Global setup
# =====================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def seed_worker(worker_id):
    # Worker gets a different, but deterministic, seed derived from the main seed
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def set_seed(seed: int, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # all GPUs

    if deterministic:
        cudnn.deterministic = True
        cudnn.benchmark = False
        # Raises error if a non-deterministic op is used; great for debugging reproducibility
        torch.use_deterministic_algorithms(True)
    else:
        cudnn.deterministic = False
        cudnn.benchmark = True

In [7]:
def save_indices(indices, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, np.array(indices, dtype=np.int64))

def load_indices(path):
    return np.load(path).tolist()

def create_or_load_subset_from_group(
    dataset,
    group_A,
    save_dir,
    subset_size=10000,
    num_classes=10,
    seed=42,
    force_rebuild=False
):
    save_path = Path(save_dir + f"/group_A_subset_{subset_size}_from_{len(group_A)}_seed{seed}.npy")

    if save_path.exists() and not force_rebuild:
        print(f"[INFO] Loading subset from {save_path}")
        return load_indices(save_path)

    print("[INFO] Creating new balanced subset from Group A")

    rng = np.random.default_rng(seed)
    n_per_class = subset_size // num_classes

    # Build class-to-indices mapping using only group_A indices
    class_to_indices = {}
    for idx in group_A:
        _, y = dataset[idx]
        class_to_indices.setdefault(y, []).append(idx)

    subset = []
    for c in range(num_classes):
        indices = np.array(class_to_indices[c])
        rng.shuffle(indices)
        subset.extend(indices[:n_per_class])

    rng.shuffle(subset)
    save_indices(subset, save_path)
    return subset

In [5]:
def main_nega_10000(seed, r, yaml_file_path): # this function is used to calculate the model from main_train_nega.py
    print(device)

    exp_yaml = process_yaml_file(yaml_file_path)
    exp_setup = process_experiment_setup(exp_yaml) 

    print('==> Loading model..')
    model_name = exp_yaml["Scenario_Name"] + f"_{seed}_{round(r,2)}"
    model_dir = './saved_models/vanilla/'
    model_folder = Path(model_dir)/model_name

    # ckpt_path = load_last_checkpoint(model_folder)
    ckpt_path = load_best_checkpoint(model_folder)
    if ckpt_path is None:
        print(f"[⚠️] No .pth files found in {model_folder}")
    
    net = exp_setup["Model"].to(device) # load model
    state = torch.load(ckpt_path, map_location=device)
    net.load_state_dict(state)

    print('==> Preparing data..')
    g = torch.Generator()
    g.manual_seed(seed)

    in_sample_set = exp_setup["Dataset"].in_sample_set # No data augmentation here
    out_sample_set =  exp_setup["Dataset"].test_set
    #train_set = exp_setup["Dataset"].train_set
    
    # Important: the in sample evaluation set should be the same each round
    group_A = create_or_load_group_A(dataset=in_sample_set, save_dir=f'./Indices/{exp_yaml['Dataset']['name']}/',
                                                  group_size=exp_setup["GroupSize"], num_classes=exp_setup["NumClasses"], seed=42, force_rebuild=False)
    subset_A = create_or_load_subset_from_group(
                dataset=in_sample_set,
                group_A=group_A,
                save_dir=f'./Indices/{exp_yaml["Dataset"]["name"]}/',
                subset_size=10000,
                num_classes=exp_setup["NumClasses"],
                seed=42,
                force_rebuild=False
            )
    
    in_sample_subset = exp_setup["Dataset"].subset("train", subset_A, clean=True)

    in_sample_loader = DataLoader(in_sample_subset,batch_size=128,shuffle=False,num_workers=8, # in sample without augmentation
                worker_init_fn=seed_worker,generator=g, persistent_workers=True, 
                pin_memory=True)
    
    out_sample_loader = DataLoader(out_sample_set,batch_size=128,shuffle=False,num_workers=8, # out sample
                worker_init_fn=seed_worker,generator=g, persistent_workers=True, 
                pin_memory=True)

    log_dir = './saved_logs/vanilla/MI_v1'
    os.makedirs(log_dir, exist_ok=True)
    log_name = model_name
    log_file = os.path.join(log_dir, f"training_log_{log_name}_MI.csv")

    if not os.path.exists(log_file):
        with open(log_file, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Scenario','Epoch','I(X;T)-InAug', 'I(T;Y)-InAug', 'I(X;T)-In', 'I(T;Y)-In','I(X;T)-Out', 'I(T;Y)-Out']) 

    # Evaluation loop
    print(f"This is the {seed} round on {r} rate 99 epoch!") # here we only consider calculating the last epoch's metric
    # value_xt_inAug, value_ty_inAug = evaluate2(net, trainloader, device)
    value_xt_in, value_ty_in = evaluate2(net, in_sample_loader, device)
    value_xt_out, value_ty_out = evaluate2(net, out_sample_loader, device)

    with open(log_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([log_name, 99,
                            0, 0, # default value for non-calculated groups
                            value_xt_in, value_ty_in,
                            value_xt_out, value_ty_out
                            ])

In [8]:
if __name__ == "__main__":
    torch.multiprocessing.set_start_method("spawn", force=True)

    # Folder containing all YAML experiment plans
    exp_dir = "./saved_exp_plan/train_plan"
    yaml_files = sorted(glob.glob(os.path.join(exp_dir, "*.yaml")))
    
    if not yaml_files:
            print(f"No YAML files found in {exp_dir}")
    else:
        print(f"Found {len(yaml_files)} experiment plan(s):")
        for f in yaml_files:
            print(" -", f)

    # Iterate over YAML files and seeds
    for yaml_path in yaml_files:
        print(f"\n========== Starting experiments from {yaml_path} ==========")
            
        for seed in range(42, 52): # entry point for main_nega
            print(f"\n>>> Running seed {seed} for {os.path.basename(yaml_path)}")
            set_seed(seed)
            
            main_nega(seed, 1.0, yaml_path)

Found 4 experiment plan(s):
 - ./saved_exp_plan/train_plan\CIFAR100_RES18_SGD.yaml
 - ./saved_exp_plan/train_plan\CIFAR100_VGG16_SGD.yaml
 - ./saved_exp_plan/train_plan\CIFAR10_RES18_SGD.yaml
 - ./saved_exp_plan/train_plan\CIFAR10_VGG16_SGD.yaml

========== Starting experiments from ./saved_exp_plan/train_plan\CIFAR100_RES18_SGD.yaml ==========

>>> Running seed 42 for CIFAR100_RES18_SGD.yaml
cuda
this time experiment blueprint:{'Scenario_Name': 'CIFAR-100_ResNet-18_40000', 'Dataset': {'name': 'CIFAR-100', 'normalization': 'cifar100', 'img_size': 32, 'group_size': 40000, 'train_transforms': [{'name': 'RandomCrop', 'params': {'size': 32, 'padding': 4}}, {'name': 'RandomHorizontalFlip', 'params': {'p': 0.5}}, {'name': 'ToTensor'}, {'name': 'Normalize'}], 'test_transforms': [{'name': 'ToTensor'}, {'name': 'Normalize'}]}, 'Model': 'ResNet-18', 'Optimizer': {'name': 'SGD', 'params': {'lr': 0.1, 'momentum': 0.9, 'nesterov': True, 'weight_decay': 0.0005}}, 'Scheduler': {'name': 'CosineAnneali

RuntimeError: DataLoader worker (pid(s) 61756, 81404, 62404, 96836, 41316, 54112, 80716, 14368) exited unexpectedly